# Notebook 01 — Feature Exploration of the Pretrained DINOv2 Backbone

This notebook corresponds to **Chapter 3** of the project report. It exercises
a frozen, pretrained DINOv2 backbone and produces the figures and quantitative
probes that Chapter 3 refers to.

**Everything here runs on frozen weights — no gradient updates.** The goal is to
*characterize* the representation we will later build our VPR system on top of.

## Outline
1. Load the pretrained DINOv2 backbone (HuggingFace `facebook/dinov2-base`)
2. **Qualitative probes**
   - Attention maps of the final block — test the "emergent segmentation" claim
   - Patch-feature PCA rendered as RGB — test semantic coherence of local features
   - Dense patch-to-patch similarity across images — foundation of EffoVPR
3. **Quantitative probes**
   - kNN classification on CIFAR-100 using frozen features.
   - Optional: linear probe on CIFAR-100
4. Save all figures and tables to `results/figures/feature_analysis/`

> **Runtime:** ~5 minutes on a single GPU (T4 or better), mostly data download + kNN extraction.

## 1. Bootstrap

In [ ]:
# -----------------------------------------------------------------
# Package bootstrap: make the repo's ``src/`` directory importable
# under the name ``dinovpr`` without requiring ``pip install -e .``
# -----------------------------------------------------------------
import sys, importlib.util
from pathlib import Path

REPO_ROOT = Path.cwd()
# If the notebook is launched from notebooks/, go up one level.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if "dinovpr" not in sys.modules:
    spec = importlib.util.spec_from_file_location(
        "dinovpr",
        REPO_ROOT / "src" / "__init__.py",
        submodule_search_locations=[str(REPO_ROOT / "src")],
    )
    m = importlib.util.module_from_spec(spec)
    sys.modules["dinovpr"] = m
    spec.loader.exec_module(m)

print("Repo root:", REPO_ROOT)

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader

# Silence a few noisy transformers/torch warnings that clutter the notebook.
warnings.filterwarnings("ignore", category=FutureWarning)

from dinovpr.data.datasets import build_cifar100
from dinovpr.data.transforms import build_eval_transform, IMAGENET_MEAN, IMAGENET_STD
from dinovpr.utils.io import get_device, set_seed
from dinovpr.utils.feature_analysis import extract_features, knn_classify, patch_pca
from dinovpr.utils.visualization import attention_heatmap, overlay_heatmap, denormalize, save_figure

set_seed(0)
DEVICE = get_device()
FIG_DIR = REPO_ROOT / "results" / "figures" / "feature_analysis"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE)
print("figures -> ", FIG_DIR)

## 2. Load the pretrained DINOv2 backbone

We use HuggingFace's `facebook/dinov2-base` (ViT-B/14, 86M parameters, patch size 14).
It exposes a standard transformers interface: forward returns `last_hidden_state`
(CLS + patch tokens) and `pooler_output` (CLS after layer-norm).

In [ ]:
from transformers import AutoModel, AutoImageProcessor

HF_ID = "facebook/dinov2-base"
model = AutoModel.from_pretrained(HF_ID).to(DEVICE).eval()
processor = AutoImageProcessor.from_pretrained(HF_ID, use_fast=True)

# Freeze (paranoia — transformers returns eval() but we never want grads)
for p in model.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"loaded {HF_ID}: {n_params:.1f}M params, hidden_size={model.config.hidden_size}, patch_size={model.config.patch_size}")

## 3. Qualitative probes

### 3.1 Attention maps of the final block

We pick a handful of natural images (either from local files or from the dataset),
forward them through DINOv2, and visualize the **CLS→patch** attention weights of
every head in the final Transformer block, overlaid on the input.

DINO's celebrated finding is that these attention maps *implicitly segment*
the salient object of the image, without any segmentation supervision.

In [ ]:
# Get a few sample images. If the user provided their own under docs/samples/
# they will be used; otherwise we pull images from CIFAR-100 for convenience.
SAMPLES_DIR = REPO_ROOT / "docs" / "samples"
sample_paths = sorted(SAMPLES_DIR.glob("*.jpg")) + sorted(SAMPLES_DIR.glob("*.png"))

if sample_paths:
    print(f"using {len(sample_paths)} local samples from {SAMPLES_DIR}")
    sample_images = [Image.open(p).convert("RGB") for p in sample_paths[:4]]
else:
    print("no local samples found; using 4 CIFAR-100 images (upscaled to 224).")
    ds = build_cifar100(
        root=str(REPO_ROOT / "data" / "cifar100"),
        train=False,
        eval_transform=build_eval_transform(image_size=224),
        download=True,
    )
    # Grab 4 diverse indices; CIFAR-100 is small so anything works.
    picks = [0, 123, 500, 1500]
    # We want PIL images for display; recover them from the underlying dataset
    import torchvision.datasets as tvd
    raw = tvd.CIFAR100(root=str(REPO_ROOT / "data" / "cifar100"), train=False, download=False)
    sample_images = [raw[i][0].convert("RGB").resize((224, 224), Image.BICUBIC) for i in picks]

print(f"loaded {len(sample_images)} sample images")

In [ ]:
# Forward pass with attention outputs enabled.
def forward_with_attention(img_pil):
    inputs = processor(images=img_pil, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)
    # Last block's attention weights: (1, n_heads, N+1, N+1)
    last_attn = out.attentions[-1][0]  # drop batch dim
    # Patch grid side = sqrt(N_patches). For ViT-B/14 @ 224, patches = 16x16 = 256.
    N_patches = last_attn.shape[-1] - 1
    grid = int(round(N_patches ** 0.5))
    return last_attn.cpu(), grid

# Build a grid: rows = images, cols = (input, mean attn, per-head attn)
n_imgs = len(sample_images)
# First collect the attention maps so we know n_heads.
attn_maps, grids = [], []
for img in sample_images:
    a, g = forward_with_attention(img)
    attn_maps.append(a); grids.append(g)

n_heads = attn_maps[0].shape[0]
fig, axes = plt.subplots(n_imgs, 1 + 1 + n_heads, figsize=(2.2 * (2 + n_heads), 2.2 * n_imgs))
if n_imgs == 1:
    axes = axes[None, :]

for r, (img, a, g) in enumerate(zip(sample_images, attn_maps, grids)):
    img_np = np.asarray(img.resize((224, 224), Image.BICUBIC))
    # Input
    axes[r, 0].imshow(img_np); axes[r, 0].set_title("input" if r == 0 else "")
    axes[r, 0].axis("off")

    # Mean attention over heads
    maps = attention_heatmap(a, grid_size=g)  # (H, g, g)
    mean_map = maps.mean(axis=0)
    axes[r, 1].imshow(overlay_heatmap(img_np, mean_map, alpha=0.6))
    axes[r, 1].set_title("mean" if r == 0 else "")
    axes[r, 1].axis("off")

    # Per-head
    for h in range(n_heads):
        axes[r, 2 + h].imshow(overlay_heatmap(img_np, maps[h], alpha=0.6))
        axes[r, 2 + h].set_title(f"head {h}" if r == 0 else "")
        axes[r, 2 + h].axis("off")

fig.suptitle(f"DINOv2 final-block CLS-attention ({HF_ID})", y=1.02)
save_figure(fig, FIG_DIR / "attention_maps.png")
plt.show()

### 3.2 Patch-feature PCA rendered as RGB

For each image, take the patch tokens of the last block, run PCA across patches,
and render the top 3 components as an RGB image. If the features are semantically
coherent, regions belonging to the same object class should map to similar colors,
and the first component alone often separates foreground from background.

In [ ]:
def get_patch_features(img_pil, take_last_n_layers: int = 1):
    """Return last-layer patch features reshaped to (grid, grid, d)."""
    inputs = processor(images=img_pil, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inputs)
    tokens = out.last_hidden_state[0]            # (N+1, d)
    patch = tokens[1:]                           # drop CLS
    N = patch.shape[0]; g = int(round(N ** 0.5))
    return patch.reshape(g, g, -1).cpu()


fig, axes = plt.subplots(len(sample_images), 3, figsize=(9, 3 * len(sample_images)))
if len(sample_images) == 1:
    axes = axes[None, :]

for r, img in enumerate(sample_images):
    img_np = np.asarray(img.resize((224, 224), Image.BICUBIC))
    patch_feats = get_patch_features(img)   # (g, g, d)

    # Foreground-mask proposal from the sign of the first PCA component.
    # (Several DINOv2 papers use this heuristic: PC1 tends to separate
    # foreground from background cleanly.)
    rgb_full = patch_pca(patch_feats, n_components=3)
    # Compute first PC sign as a rough foreground mask
    pc1 = rgb_full[..., 0]
    fg_mask = torch.from_numpy(pc1 > pc1.mean())  # (g, g) bool
    rgb_fg = patch_pca(patch_feats, n_components=3, foreground_mask=fg_mask)

    axes[r, 0].imshow(img_np); axes[r, 0].set_title("input" if r == 0 else ""); axes[r, 0].axis("off")
    # Upsample PCA to image size for visualization
    def up(rgb):
        t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).float()
        t = F.interpolate(t, size=(224, 224), mode="nearest")
        return t[0].permute(1, 2, 0).numpy()
    axes[r, 1].imshow(up(rgb_full)); axes[r, 1].set_title("PCA (all patches)" if r == 0 else "")
    axes[r, 1].axis("off")
    axes[r, 2].imshow(up(rgb_fg));   axes[r, 2].set_title("PCA (foreground-masked)" if r == 0 else "")
    axes[r, 2].axis("off")

fig.suptitle("Top-3 PCA of DINOv2 patch features, rendered as RGB", y=1.02)
save_figure(fig, FIG_DIR / "patch_pca.png")
plt.show()

### 3.3 Dense patch-to-patch similarity across images

For a user-selected query patch in image A, compute the cosine similarity of
its feature with every patch in image B, and display the heatmap. This is the
exact mechanism that training-free VPR methods like EffoVPR exploit for
re-ranking — nearest-neighbour matching of internal patch descriptors.

In [ ]:
def dense_similarity(img_a, img_b, query_xy=(112, 50)):
    """Return a similarity heatmap of size (224, 224) for every patch of B
    against the single patch of A containing pixel `query_xy`."""
    feats_a = get_patch_features(img_a)  # (g, g, d)
    feats_b = get_patch_features(img_b)  # (g, g, d)
    g = feats_a.shape[0]
    patch_px = 224 // g
    qx = min(query_xy[0] // patch_px, g - 1)
    qy = min(query_xy[1] // patch_px, g - 1)
    q = feats_a[qy, qx]                  # (d,)
    q = F.normalize(q, dim=-1)
    fb = F.normalize(feats_b, dim=-1)    # (g, g, d)
    sim = (fb @ q).numpy()               # (g, g)
    return sim, (qx * patch_px + patch_px // 2, qy * patch_px + patch_px // 2), g


# Demonstrate on 3 pairs: image i matched against image (i+1) % n
n_pairs = min(3, len(sample_images))
fig, axes = plt.subplots(n_pairs, 2, figsize=(7, 3.2 * n_pairs))
if n_pairs == 1:
    axes = axes[None, :]

for r in range(n_pairs):
    a = sample_images[r]
    b = sample_images[(r + 1) % len(sample_images)]
    a_np = np.asarray(a.resize((224, 224), Image.BICUBIC))
    b_np = np.asarray(b.resize((224, 224), Image.BICUBIC))
    sim, (cx, cy), g = dense_similarity(a, b, query_xy=(112, 112))

    axes[r, 0].imshow(a_np)
    axes[r, 0].scatter([cx], [cy], marker="+", s=200, c="red", linewidths=3)
    axes[r, 0].set_title("query (red cross)" if r == 0 else "")
    axes[r, 0].axis("off")

    axes[r, 1].imshow(overlay_heatmap(b_np, sim, alpha=0.55))
    axes[r, 1].set_title("similarity to query" if r == 0 else "")
    axes[r, 1].axis("off")

fig.suptitle("Cross-image dense patch similarity (foundation of EffoVPR-style re-ranking)", y=1.02)
save_figure(fig, FIG_DIR / "patch_similarity.png")
plt.show()

## 4. Quantitative probes

### 4.1 kNN classification on CIFAR-100

Extract frozen DINOv2 global features (pooler output) for the full CIFAR-100
train and test splits, then run weighted kNN classification (cosine similarity,
k=20). This number is the main empirical sanity check of "are the pretrained
features actually useful out of the box".

> Note: CIFAR-100 images are 32×32 but DINOv2 expects 224×224. We upsample with
> bicubic interpolation. This is nonstandard for CIFAR classification but is
> the correct protocol when probing an ImageNet-scale backbone.

In [ ]:
BATCH = 128
NUM_WORKERS = 2

cifar_root = str(REPO_ROOT / "data" / "cifar100")
eval_tf = build_eval_transform(image_size=224)

train_ds = build_cifar100(root=cifar_root, train=True,  eval_transform=eval_tf, download=True)
val_ds   = build_cifar100(root=cifar_root, train=False, eval_transform=eval_tf, download=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

def cls_extractor(x):
    return model(pixel_values=x).pooler_output

print("Extracting features ...")
train_feats, train_labels = extract_features(cls_extractor, train_loader, DEVICE, desc="train")
val_feats,   val_labels   = extract_features(cls_extractor, val_loader,   DEVICE, desc="val")
print(f"train: {train_feats.shape} | val: {val_feats.shape}")

In [ ]:
results = knn_classify(
    train_feats, train_labels, val_feats, val_labels,
    num_classes=100, k=20, temperature=0.07,
)
print(f"kNN (k=20) on CIFAR-100 with frozen {HF_ID}:")
print(f"  top-1 = {results['top1']:.2f}%")
print(f"  top-5 = {results['top5']:.2f}%")

# Save summary as CSV for the report.
import csv
tab_path = REPO_ROOT / "results" / "tables" / "feature_analysis_knn.csv"
tab_path.parent.mkdir(parents=True, exist_ok=True)
with tab_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["backbone", "dataset", "k", "top1", "top5"])
    w.writerow([HF_ID, "cifar100", 20, f"{results['top1']:.2f}", f"{results['top5']:.2f}"])
print("saved", tab_path)

### 4.2 (Optional) Linear probe

A linear probe is a stronger indicator of feature quality than kNN because it
can learn a per-class decision boundary. It also takes noticeably longer
(100 epochs of SGD on the train-set features). Uncomment the cell below to
run it.

In [ ]:
# Uncomment to run:
# from dinovpr.utils.feature_analysis import linear_probe
# res_linear = linear_probe(
#     train_feats, train_labels, val_feats, val_labels,
#     num_classes=100, epochs=100, batch_size=512, lr=0.01,
# )
# print(f"Linear probe: top-1={res_linear['top1']:.2f}%  top-5={res_linear['top5']:.2f}%")

## 5. Summary

We have:

- Verified the **emergent-segmentation** property of DINOv2's final-layer
  CLS attention.
- Observed that **patch-feature PCA** produces semantically coherent
  foreground/background separations.
- Confirmed that **dense patch-to-patch similarity** generalises across
  images — the foundation of training-free VPR re-ranking methods
  like EffoVPR.
- Measured a **quantitative baseline** via kNN classification on CIFAR-100
  with a frozen backbone.

These observations inform our design choices in Part III:
1. We will use DINOv2 as our **frozen VPR backbone** — its out-of-the-box features
   are already strong.
2. For re-ranking, we will rely on **internal attention features** (EffoVPR-style),
   since the dense similarity structure above shows these are meaningful.
3. For any adaptation, we will prefer **PEFT methods** (LoRA) over full
   fine-tuning — the frozen features are already so good that catastrophic
   forgetting from full fine-tuning is a real risk.

The corresponding report chapter (Chapter 3) will be updated with the figures
saved under `results/figures/feature_analysis/` and the kNN numbers in
`results/tables/feature_analysis_knn.csv`.